In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

In [3]:
# 1. Data Preparation
df = pd.read_csv(r"C:\Users\sinha\OneDrive\Documents\Projects\Music Recommadation System on Machine Learning\Music-Recommendation-System\data\ex.csv")
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df['User-Rating'] = df['User-Rating'].str[:3]
df['Album/Movie'] = df['Album/Movie'].str.replace(' ', '')
df['Singer/Artists'] = df['Singer/Artists'].str.replace(',', '  ')
df['tags'] = (df['Singer/Artists'] + ' ' + df['Genre'] + ' ' + df['Album/Movie'] + ' ' + df['User-Rating']).str.lower()
new_df = df[['Song-Name', 'tags']].rename(columns={'Song-Name': 'title'})



In [ ]:
# 2. Train-Test Split
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(new_df, test_size=0.2, random_state=42)



In [5]:
# 3. TF-IDF + Cosine Similarity Model
tfidf = TfidfVectorizer(max_features=2000)
tfidf_vectors = tfidf.fit_transform(train_df['tags']).toarray()
tfidf_similarity = cosine_similarity(tfidf_vectors)



In [6]:
def recommend_tfidf(song_title, train_df=train_df, similarity=tfidf_similarity, top_n=5):
    try:
        idx = train_df[train_df['title'] == song_title].index[0]
        sim_scores = list(enumerate(similarity[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
        recommended = [train_df.iloc[i[0]].title for i in sim_scores]
        return recommended
    except IndexError:
        return []



In [ ]:
import os

# Ensure the 'model' directory exists
os.makedirs('model', exist_ok=True)

# 4. KNN Model
knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(tfidf_vectors)
_, indices = knn.kneighbors(tfidf_vectors)

def recommend_knn(song_title, train_df=train_df, knn_indices=indices, top_n=5):
    try:
        idx = train_df[train_df['title'] == song_title].index[0]
        recommended = [train_df.iloc[i].title for i in knn_indices[idx][1:top_n+1]]
        return recommended
    except IndexError:
        return []

# 5. Simple Evaluation (Precision@5)
def precision_at_k(recommend_func, test_df, train_df, k=5):
    hits = 0
    total = 0
    for song in test_df['title']:
        recs = recommend_func(song)
        if song in recs:
            hits += 1
        total += 1
    return hits / total if total > 0 else 0

precision_tfidf = precision_at_k(lambda x: recommend_tfidf(x, train_df, tfidf_similarity), test_df, train_df)
precision_knn = precision_at_k(lambda x: recommend_knn(x, train_df, indices), test_df, train_df)

print(f"TF-IDF + Cosine Similarity Precision@5: {precision_tfidf:.2f}")
print(f"KNN Precision@5: {precision_knn:.2f}")

# 6. Select and Save Best Model


import pickle

if precision_tfidf >= precision_knn:
    pickle.dump(tfidf, open('model/tfidf_vectorizer.pkl', 'wb'))
    pickle.dump(tfidf_vectors, open('model/tfidf_vectors.pkl', 'wb'))
    pickle.dump(train_df, open('model/musicrec_train_df.pkl', 'wb'))
    pickle.dump(tfidf_similarity, open('model/tfidf_similarity.pkl', 'wb'))
else:
    print("Best algorithm: KNN")
    pickle.dump(knn, open('model/knn_model.pkl', 'wb'))
    pickle.dump(tfidf_vectors, open('model/tfidf_vectors.pkl', 'wb'))
    pickle.dump(train_df, open('model/musicrec_train_df.pkl', 'wb'))
    pickle.dump(indices, open('model/knn_indices.pkl', 'wb'))


def load_and_recommend(song_title, top_n=5):
    train_df = pickle.load(open('model/musicrec_train_df.pkl', 'rb'))
    tfidf = pickle.load(open('model/tfidf_vectorizer.pkl', 'rb'))
    tfidf_vectors = pickle.load(open('model/tfidf_vectors.pkl', 'rb'))
    tfidf_similarity = pickle.load(open('model/tfidf_similarity.pkl', 'rb'))
    try:
        idx = train_df[train_df['title'] == song_title].index[0]
        sim_scores = list(enumerate(tfidf_similarity[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
        recommended = [train_df.iloc[i[0]].title for i in sim_scores]
        return recommended
    except IndexError:
        return []

TF-IDF + Cosine Similarity Precision@5: 0.00
KNN Precision@5: 0.00
Best algorithm: TF-IDF + Cosine Similarity


In [ ]:
import os, pickle
os.makedirs('model', exist_ok=True)
pickle.dump(train_df, open('model/musicrec_train_df.pkl', 'wb'))
